In [19]:
!pip install transformers torch --quiet

In [20]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

print("Libraries imported successfully!")

Libraries imported successfully!


In [21]:
# Define the model name from Hugging Face Model Hub
MODEL_NAME = "microsoft/DialoGPT-medium"

print(f"Loading tokenizer from: {MODEL_NAME} ...")
# Load tokenizer — converts text ↔ token IDs
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, padding_side='left')

print(f"Loading model from: {MODEL_NAME} ...")
# Load pre-trained causal language model
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)

# Set the model to evaluation mode (disables dropout, etc.)
model.eval()

print("\nModel and tokenizer loaded successfully!")

Loading tokenizer from: microsoft/DialoGPT-medium ...
Loading model from: microsoft/DialoGPT-medium ...


Loading weights:   0%|          | 0/293 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
GPT2LMHeadModel LOAD REPORT from: microsoft/DialoGPT-medium
Key                              | Status     |  | 
---------------------------------+------------+--+-
transformer.h.{0...23}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Model and tokenizer loaded successfully!


In [22]:
def generate_response(user_input, chat_history_ids=None, max_history_length=1000):
    # Encode user input with EOS token
    new_input_ids = tokenizer.encode(
        user_input + tokenizer.eos_token,
        return_tensors='pt'
    )

    # Build attention mask for new input (1 = real token, not padding)
    new_attention_mask = torch.ones_like(new_input_ids)

    # Concatenate with conversation history if it exists
    if chat_history_ids is not None:
        if chat_history_ids.shape[-1] > max_history_length:
            chat_history_ids = chat_history_ids[:, -max_history_length:]
        bot_input_ids = torch.cat([chat_history_ids, new_input_ids], dim=-1)
        # Attention mask covers all tokens (history + new input)
        attention_mask = torch.ones_like(bot_input_ids)
    else:
        bot_input_ids = new_input_ids
        attention_mask = new_attention_mask

    # Generate response
    with torch.no_grad():
        chat_history_ids = model.generate(
            bot_input_ids,
            attention_mask=attention_mask,       # ← fixes the warning
            max_new_tokens=200,
            pad_token_id=tokenizer.eos_token_id,
            do_sample=True,
            top_k=50,
            top_p=0.95,
            temperature=0.75,
            repetition_penalty=1.3,
        )

    # Decode only the newly generated tokens
    response = tokenizer.decode(
        chat_history_ids[:, bot_input_ids.shape[-1]:][0],
        skip_special_tokens=True
    )

    return response, chat_history_ids

In [ ]:
def run_chatbot():
    """
    Run an interactive console-based chatbot loop.
    Type 'exit' or 'quit' to end the conversation.
    """


    print("   AI Chatbot powered by DialoGPT (Hugging Face)")

    print("Chatbot: Hello! I am your AI assistant. How can I help you today?")


    # Initialise conversation history as None (empty at start)
    chat_history_ids = None

    # Main conversation loop
    while True:
        # Get input from the user
        user_input = input("You: ").strip()

        # Skip empty inputs gracefully
        if not user_input:
            print("Chatbot: It seems like you didn't type anything. Please go ahead!")
            continue

        # Exit condition — stop chatbot when user types 'exit' or 'quit'
        if user_input.lower() in ["exit", "quit"]:
            print("Chatbot: Thank you for chatting! Have a great day. Goodbye!")

            break

        # Generate a response from the model
        response, chat_history_ids = generate_response(user_input, chat_history_ids)

        # Display the chatbot's response
        print(f"Chatbot: {response}")



# Start the chatbot
run_chatbot()

   AI Chatbot powered by DialoGPT (Hugging Face)
Chatbot: Hello! I am your AI assistant. How can I help you today?
You: How are you?
Chatbot: Good, how bout yourself?
You: I am good
